Dataset Link: https://www.kaggle.com/datasets/uciml/glass

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import cdist
from sklearn.cluster import DBSCAN


%matplotlib inline

In [2]:
df = pd.read_csv("glass.csv")

df.head(3)

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1


In [3]:
X = df.drop(columns=["Type"], errors='ignore')

In [4]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Statistical Method

In [5]:
# 1. Z Score Method
z_scores = np.abs((X - X.mean()) / X.std())
outliers_z = (z_scores > 3).any(axis=1)

print("Z-score outliers:", outliers_z.sum())

Z-score outliers: 20


In [6]:
# 2. IQR Method
Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)
IQR = Q3 - Q1

outliers_iqr = ((X < (Q1 - 1.5 * IQR)) | (X > (Q3 + 1.5 * IQR))).any(axis=1)

print("IQR outliers:", outliers_iqr.sum())

IQR outliers: 78


# Distance Based Method

In [7]:
# 1. KNN Algorithm
k = 5
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_scaled)

distances, _ = nn.kneighbors(X_scaled)

# Distance to k-th neighbor
knn_dist = distances[:, -1]

# Threshold (top 5% as outliers)
threshold = np.percentile(knn_dist, 95)
outliers_knn = knn_dist > threshold

print("KNN outliers:", outliers_knn.sum())

KNN outliers: 11


In [8]:
# 2. DISTANCE THRESHOLD METHOD
# Distance from mean
mean_point = np.mean(X_scaled, axis=0)
distances = cdist(X_scaled, [mean_point])

threshold = np.percentile(distances, 95)
outliers_dist = distances.flatten() > threshold

print("Distance threshold outliers:", outliers_dist.sum())

Distance threshold outliers: 11


# Density Based Method

In [9]:
# DBSCAN (Density-Based)
dbscan = DBSCAN(eps=1.5, min_samples=5)
labels = dbscan.fit_predict(X_scaled)

# -1 = outliers
outliers_dbscan = labels == -1

print("DBSCAN outliers:", outliers_dbscan.sum())

DBSCAN outliers: 41
